# 02. Write와 Select

Write는 정보를 외부 기록에 남기는 단계이고, Select는 현재 질문에 필요한 기록만 고르는 단계입니다.

## 실행 환경 준비

첫 코드 셀은 노트북에서 사용할 패키지 버전을 맞춥니다. 설치가 끝나면 이어지는 셀을 순서대로 실행합니다.

In [1]:
# %pip은 현재 노트북 커널에 패키지를 설치하는 Jupyter 전용 명령입니다.
# -q는 설치 로그를 줄이고, == 뒤 숫자는 수업에서 검증한 버전을 고정합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


## 1. API 연결 준비

각 노트북은 빈 커널에서 시작할 수 있도록 연결 코드를 포함합니다.

In [2]:
# find_dotenv는 현재 폴더부터 상위 폴더를 검색해 .env 경로를 찾습니다.
# model은 이후 셀에서 공통으로 사용할 LangChain 채팅 모델 객체입니다.
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model
import os

# usecwd=True는 노트북을 실행한 폴더를 검색 시작점으로 지정합니다.
# 키 값은 출력하지 않고 존재 여부만 확인합니다.
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
# openai:는 공급자 이름이며, reasoning_effort="none"은 별도 추론 토큰을 사용하지 않는 설정입니다.
# max_tokens=300은 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model("openai:gpt-5.4-mini", reasoning_effort="none", max_tokens=300)
print("LangChain 모델 준비 완료")

LangChain 모델 준비 완료


## 2. 기록을 JSON으로 저장하고 다시 읽습니다

예제 데이터는 짧게 인라인으로 두고, 임시 폴더에 저장하여 작업 폴더에 의존하지 않게 합니다.

In [3]:
# records의 각 사전은 문서 식별자, 주제, 본문을 나타냅니다.
# 임시 JSON 파일에 썼다가 다시 읽어 외부 기록의 저장과 불러오기를 확인합니다.
import json
from pathlib import Path
from tempfile import gettempdir

records = [
    {"id": "E01-p2", "topic": "휴관", "text": "매주 월요일 휴관"},
    {"id": "E01-p3", "topic": "대출", "text": "7권, 14일 대출"},
    {"id": "E01-p5", "topic": "주차", "text": "90분 무료"},
]
memory_path = Path(gettempdir()) / "ch08-memory.json"
memory_path.write_text(json.dumps(records, ensure_ascii=False), encoding="utf-8")
loaded = json.loads(memory_path.read_text(encoding="utf-8"))
print("다시 읽은 기록:", len(loaded))

다시 읽은 기록: 3


## 3. 질문에 필요한 기록만 선택합니다

여기서는 학생이 식별자를 직접 지정합니다. 자동 검색은 Ch 9에서 다룹니다.

In [4]:
# selected_ids는 이번 질문에 필요한 문서 ID 모음입니다.
# 리스트 컴프리헨션은 loaded를 순회하며 ID가 포함된 기록만 selected에 남깁니다.
selected_ids = {"E01-p3", "E01-p5"}
selected = [row for row in loaded if row["id"] in selected_ids]
for row in selected:
    print(row["id"], row["topic"], row["text"])

E01-p3 대출 7권, 14일 대출
E01-p5 주차 90분 무료


## 4. 선택한 기록으로 답을 생성합니다

모델에는 저장소 전체 대신 선택한 두 기록만 전달합니다.

In [5]:
# join은 선택한 기록을 줄바꿈으로 합치고, f-string은 각 근거 앞에 출처 ID를 붙입니다.
# prompt는 합친 근거와 학생 질문을 모델에 전달할 하나의 문자열입니다.
context = "\n".join(f"[{r['id']}] {r['text']}" for r in selected)
prompt = f"""근거:
{context}

질문: 대출 권수와 주차 무료 시간을 알려 주세요."""

In [6]:
# messages는 역할과 내용을 짝지은 목록이며 system 지시를 user 질문보다 먼저 둡니다.
# invoke()는 모델을 한 번 호출하고, 반환된 AIMessage에서 text와 사용량을 읽습니다.
messages = [
    ("system", "제공된 근거만 사용하고 문장마다 식별자를 붙이세요."),
    ("user", prompt),
]
response = model.invoke(messages)
# usage_metadata는 입력·출력 토큰 수를 담지만 제공되지 않을 가능성도 검사합니다.
usage = response.usage_metadata
if usage is None:
    raise RuntimeError("모델 사용량 메타데이터가 없습니다.")
print(response.text)
print("input_tokens:", usage.get("input_tokens"))
print("output_tokens:", usage.get("output_tokens"))

대출 권수는 7권입니다.[E01-p3]  
주차 무료 시간은 90분입니다.[E01-p5]
input_tokens: 71
output_tokens: 34


## 정리

외부 기록은 저장만으로 모델에 전달되지 않습니다. 질문에 맞는 항목을 선택하여 호출 입력에 넣어야 합니다.

## 실패 재현: 저장한 기록을 선택하지 않습니다

파일에 기록이 있어도 선택 결과가 비어 있으면 모델에 전달할 근거가 없습니다.

In [7]:
# rows는 선택된 근거 목록이며, 빈 목록이면 모델에 줄 정보가 없습니다.
# None 반환 함수의 -> None과 예상 오류를 포착하는 try/except를 함께 확인합니다.
def require_context(rows: list[dict[str, str]]) -> None:
    if not rows:
        raise ValueError("선택한 근거가 없습니다.")

try:
    require_context([])
except ValueError as error:
    print("관찰한 실패:", error)
else:
    raise AssertionError("빈 근거 오류가 발생해야 합니다.")

관찰한 실패: 선택한 근거가 없습니다.


## 수정: 선택한 기록을 검사합니다

앞에서 선택한 대출과 주차 기록을 검사한 뒤 모델 입력에 사용합니다.

In [8]:
# 앞에서 만든 selected를 검사해 빈 근거 문제가 해결됐는지 확인합니다.
# 리스트 컴프리헨션으로 실제 선택 순서까지 검증합니다.
require_context(selected)
assert [row["id"] for row in selected] == ["E01-p3", "E01-p5"]
print("수정 후 선택한 기록:", len(selected))

수정 후 선택한 기록: 2
